In [1]:
import boto3
import glob
import logging
import os
import s3fs

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from dotenv import load_dotenv

# Env
load_dotenv("../.env", override=True)
REPO_ROOT = os.environ["INSTALL_PATH"]
MINIO_KEY = os.environ["MINIO_KEY"]
MINIO_SECRET = os.environ["MINIO_SECRET"]

# Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s:%(name)s:%(message)s"
)
logger = logging.getLogger(__name__)

In [2]:
# ============================================================
#       Args
# ============================================================

FILE_DRT3B_UNIPROT = f"{REPO_ROOT}/data_sync/uniprotkb_drt3b_2026_08_24.tsv"

# MinIO
BUCKET = "iei-project"
PREFIX_GOLD = "03_gold/defense_finder/"
PREFIX_SILVER = "02_silver/defense_finder/"
FILE_COMPOSITE = "composite_score.parquet/part-00000-ac5574c4-cf3c-48a9-98b6-077d4db394f9-c000.snappy.parquet"

# Dirs
OUT_DIR_PLOT = f"{REPO_ROOT}/vis/plots"

# Check live objects in MinIO silver
client = boto3.client(
    "s3",
    endpoint_url="http://eagle.tcag.ca:9000",
    aws_access_key_id=MINIO_SECRET,
    aws_secret_access_key=MINIO_KEY,
)
response = client.list_objects_v2(
    Bucket=BUCKET,
    Prefix=PREFIX_GOLD
)
live_objects = [
    obj["Key"]
    for obj in response.get("Contents", [])
]
live_objects

['03_gold/defense_finder/composite_score.parquet/_SUCCESS',
 '03_gold/defense_finder/composite_score.parquet/part-00000-ac5574c4-cf3c-48a9-98b6-077d4db394f9-c000.snappy.parquet',
 '03_gold/defense_finder/defense_human_domain_annotated.parquet',
 '03_gold/defense_finder/final_output_spark.parquet/_SUCCESS',
 '03_gold/defense_finder/final_output_spark.parquet/part-00000-bb778272-db05-479f-a79a-2d18e89fe363-c000.snappy.parquet',
 '03_gold/defense_finder/griid_gene_subset.parquet',
 '03_gold/defense_finder/human_bacteria_structural_analogs_enriched.parquet',
 '03_gold/defense_finder/human_bacteria_structural_analogs_with_scores.parquet/_SUCCESS',
 '03_gold/defense_finder/human_bacteria_structural_analogs_with_scores.parquet/part-00000-275fe83c-840b-476e-bf63-c47165847863-c000.snappy.parquet']

In [3]:
# ============================================================
#       In
# ============================================================

df_comp_score = pl.read_parquet(
    f"s3://{BUCKET}/{PREFIX_GOLD}{FILE_COMPOSITE}",
    storage_options={
        "aws_endpoint_url": "http://eagle.tcag.ca:9000",
        "aws_access_key_id": MINIO_SECRET,
        "aws_secret_access_key": MINIO_KEY,
    }
)
df_drt3b = pl.read_csv(
    FILE_DRT3B_UNIPROT,
    separator="\t",
    has_header=True
)
df_drt3b

Entry,Reviewed,Entry Name,Protein names,Gene Names,Organism,Length
str,str,str,str,str,str,i64
"""A0A059DVQ8""","""unreviewed""","""A0A059DVQ8_9PROT""","""Reverse transcriptase domain-c…","""HY36_11665""","""Hyphomonas atlantica corrig""",643
"""A0A074KSF0""","""unreviewed""","""A0A074KSF0_9BACT""","""Reverse transcriptase domain-c…","""EL17_21035""","""Anditalea andensis""",653
"""A0A096AWC2""","""unreviewed""","""A0A096AWC2_9BACT""","""DNA polymerase""","""HMPREF9302_09010""","""Prevotella amnii DNF00058""",729
"""A0A0A0EGA9""","""unreviewed""","""A0A0A0EGA9_9RHOB""","""Reverse transcriptase domain-c…","""ATO9_11360""","""Pseudooceanicola atlanticus""",529
"""A0A0C4WJG8""","""unreviewed""","""A0A0C4WJG8_9GAMM""","""Reverse transcriptase (RNA-dep…","""Achr_5060""","""Azotobacter chroococcum NCIMB …",652
…,…,…,…,…,…,…
"""Q63XF5""","""unreviewed""","""Q63XF5_BURPS""","""Reverse transcriptase domain-c…","""BPSL0582""","""Burkholderia pseudomallei (str…",690
"""Q6D8B7""","""unreviewed""","""Q6D8B7_PECAS""","""Reverse transcriptase domain-c…","""ECA1057""","""Pectobacterium atrosepticum (s…",669
"""S0F461""","""unreviewed""","""S0F461_9BACT""","""Reverse transcriptase domain-c…","""BACCOPRO_00140""","""Phocaeicola coprophilus DSM 18…",611


In [4]:
df_comp_score

defense_uniprot_ac,human_entryId,composite_score,foldseek_evalue,pymol_rmsd,tm_score_human,tm_score_bacteria,plddt_human,plddt_bacteria
str,str,f64,f64,f32,f32,f32,f32,f32
"""A0A5C5QGP9""","""A0A024R9P6""",0.2646,0.000087,null,null,null,null,83.480003
"""A0A2R3IRC4""","""A0A0D9SF92""",0.8315,1.2070e-7,0.93,0.8169,0.1028,83.199997,77.449997
"""A0A4D8PF33""","""A0A140VK70""",0.2883,0.003094,15.01,0.2908,0.1539,80.290001,83.050003
"""A0A7S9D461""","""A0A140VK70""",0.8986,5.6920e-19,2.18,0.535,0.7678,80.290001,89.629997
"""A0A2K9LJD6""","""A0A1B0GVC6""",0.2795,0.007862,10.88,0.2428,0.29,67.610001,91.379997
…,…,…,…,…,…,…,…,…
"""A0A1D7XM13""","""Q9H4E3""",0.6409,0.000001,4.93,0.6556,0.327,84.879997,82.360001
"""A0A7D6CQE3""","""Q9H4E3""",0.5814,0.000001,5.77,0.6733,0.3337,84.879997,75.230003
"""A0A5P3ALB7""","""Q9H4E3""",0.8713,1.2930e-15,2.68,0.739,0.4786,84.879997,89.019997
